# U-Net Stability Experiments 2
**WorldView-3 | 16-bit 4-band (R,G,B,NIR1) | 512×512 chips | H100 NVL**

Stability experiments testing loss function and learning rate variations.
All experiments use fold 1 of a 5-fold split, WD=1e-3, warm-up cosine scheduler.

**NOTE: Do not re-run — results saved to:**
- `results/unet_training_v3/sweep_lowwd_test/` (Cell 2)
- `results/unet_training_v3/sweep_bce_test/` (Cell 3)
- `results/unet_training_v3/sweep_lr1e4_test/` (Cell 4)

- Cell 1: WaterChipDatasetV3 — Full Augmentation (final version)
- Cell 2: Stability Test 1 — Lower LR (4e-4) + Tversky
- Cell 3: Stability Test 2 — BCE Loss
- Cell 4: Stability Test 3 — LR=1e-4 + Focal+Dice (winner → used in final sweep)

In [2]:
# ── Cell 1: WaterChipDatasetV3 — On-the-Fly Augmentation (FINAL) ──
# Loads from ORIGINAL chips only (chips_images/images/).
# Applies V3 augmentation suite randomly each time a chip is loaded.
# No pre-generated dataset needed — variety comes from random augmentation every epoch.

import torch, random, cv2
import numpy as np
import rasterio
from pathlib import Path
from torch.utils.data import Dataset

class WaterChipDatasetV3(Dataset):
    def __init__(self, chip_paths, masks_dir, mean, std, augment=False):
        self.chip_paths = chip_paths
        self.masks_dir  = Path(masks_dir)
        self.mean = torch.tensor(mean).view(-1,1,1)
        self.std  = torch.tensor(std).view(-1,1,1)
        self.augment = augment

    def __len__(self): return len(self.chip_paths)

    def __getitem__(self, idx):
        cp = self.chip_paths[idx]
        mp = self.masks_dir / cp.name
        with rasterio.open(cp) as src:
            image = np.stack([
                src.read(5), src.read(3), src.read(2), src.read(7)
            ]).astype(np.float32)
        with rasterio.open(mp) as src:
            mask = (src.read(1) > 0).astype(np.float32)

        if self.augment:
            image, mask = self._augment(image, mask)

        image = (torch.from_numpy(image) - self.mean) / self.std
        return image, torch.from_numpy(mask.copy()).unsqueeze(0)

    def _augment(self, img, mask):
        # ── Spatial — applied to image AND mask ──
        if random.random() > 0.5:
            img  = img[:, :, ::-1].copy()
            mask = mask[:, ::-1].copy()
        if random.random() > 0.5:
            img  = img[:, ::-1, :].copy()
            mask = mask[::-1, :].copy()
        if random.random() > 0.25:
            k    = random.choice([1, 2, 3])
            img  = np.rot90(img,  k, axes=(1, 2)).copy()
            mask = np.rot90(mask, k, axes=(0, 1)).copy()
        if random.random() > 0.5:
            h, w      = img.shape[1], img.shape[2]
            crop_size = random.randint(int(h * 0.7), h)
            row       = random.randint(0, h - crop_size)
            col       = random.randint(0, w - crop_size)
            img       = img[:, row:row+crop_size, col:col+crop_size]
            mask      = mask[row:row+crop_size, col:col+crop_size]
            img  = np.stack([cv2.resize(img[b], (w, h), interpolation=cv2.INTER_LINEAR) for b in range(img.shape[0])])
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

        # ── Radiometric — image only ──
        if random.random() > 0.5:
            for b in range(img.shape[0]):
                br = np.random.uniform(0.8, 1.2)
                ct = np.random.uniform(0.8, 1.2)
                mv = img[b].mean()
                img[b] = (img[b] - mv) * ct + mv * br
        if random.random() > 0.5:
            for b in range(img.shape[0]):
                noise_std = np.random.uniform(0.005, 0.03) * img[b].std()
                img[b]    = img[b] + np.random.normal(0, noise_std, img[b].shape).astype(np.float32)

        # ── Band-specific ──
        if random.random() > 0.7:
            img[random.randint(0, img.shape[0] - 1)] = 0.0

        # ── Terrain ──
        if random.random() > 0.7:
            h, w   = img.shape[1], img.shape[2]
            sh     = random.randint(h // 8, h // 3)
            sw     = random.randint(w // 8, w // 3)
            sr     = random.randint(0, h - sh)
            sc     = random.randint(0, w - sw)
            img[:, sr:sr+sh, sc:sc+sw] *= np.random.uniform(0.3, 0.7)
        if random.random() > 0.8:
            h, w   = img.shape[1], img.shape[2]
            gh     = random.randint(h // 16, h // 6)
            gw     = random.randint(w // 16, w // 6)
            gr     = random.randint(0, h - gh)
            gc     = random.randint(0, w - gw)
            img[:, gr:gr+gh, gc:gc+gw] *= np.random.uniform(1.3, 2.0)

        img = np.clip(img, 0, img.max() if img.max() > 0 else 1)
        return img, mask

print('WaterChipDatasetV3 defined — ON-THE-FLY augmentation')
print('Loads from original chips only (no pre-generated dataset)')
print('Augmentations applied randomly each time a chip is loaded')

WaterChipDatasetV3 defined — ON-THE-FLY augmentation
Loads from original chips only (no pre-generated dataset)
Augmentations applied randomly each time a chip is loaded


In [3]:
# ── Cell 2: Stability Test 1 — Lower LR (4e-4) + Tversky ──
# NOTE: Do not re-run — results saved to results/unet_training_v3/sweep_lowwd_test/
#
# Based on evidence from 3 prior runs (original, reduced-aug, diagnostic):
# ALL configurations collapsed into a near-zero-probability degenerate state,
# always at or shortly after LR reached its peak (0.001). This points to
# LR=0.001 being too aggressive for this Dice-loss + WD=0.1 combination,
# regardless of augmentation intensity.
#
# This run tests two changes simultaneously (given time constraints):
# 1. Lower peak LR: 1e-3 -> 4e-4 (60% reduction)
# 2. Tversky loss instead of Dice (different gradient behaviour on imbalanced data,
#    alpha=0.3/beta=0.7 favours recall, may be more stable than Dice here)
#
# Same warm-up (3 epochs), same smoothing (smooth=1.0), same on-the-fly augmentation
# (original intensity, since reduced-aug did NOT prevent collapse either).

import segmentation_models_pytorch as smp
import torch, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import KFold
from collections import defaultdict
from tqdm import tqdm
import rasterio

BASE_DIR   = Path('/mnt/batch/tasks/shared/LS_root/mounts/clusters/v-lmarotti1/code/Users/v-lmarotti/OmniWaterMask')
CHIPS_DIR  = BASE_DIR / 'images/chips_images/images'
MASKS_DIR  = BASE_DIR / 'images/chips_masks/masks'
OUTPUT_DIR = BASE_DIR / 'results/unet_training_v3/sweep_lowwd_test'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IN_CHANNELS   = 4
MAX_EPOCHS    = 100
PATIENCE      = 15
SEED          = 42
N_FOLDS       = 5
LAND_PCT      = 0.20
WARMUP_EPOCHS = 3
DIAG_THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5]

random.seed(SEED)
np.random.seed(SEED)

with open(BASE_DIR / 'results/unet_training/normalisation_stats.json') as f:
    stats = json.load(f)
mean = np.array(stats['mean'], dtype=np.float32)
std  = np.array(stats['std'],  dtype=np.float32)

TEST_SCENES = {
    '12AUG30193433_sub2', '15JUL02193_sub_2', '16Sep2219_1b_Chiwawa',
    '16Sep2219_1b_Teanaway', 'UGR_2017_08_30_103001007046C600_AOI_1',
    'UGR_2021_07_21_10300100C2972100_AOI_1', 'Wenatchee_2014_08_27_10300100354F7A00_AOI',
    'Wenatchee_2016_08_16_10400100214F5200_AOI_1', 'Wenatchee_2019_07_26_104001004E554F00_AOI_1',
    'Wenatchee_2022_11_09_104001007E867900_AOI_1', 'Wenatchee_2025_02_07_103001010D0A1600_AOI_1',
    'cath_creek_2015_10_09_10300100496A4600_AOI_1', 'cath_creek_2018_12_05_1040010046BF9000_AOI_1',
}

def get_scene_id(filename):
    name = Path(filename).stem
    if '_clipped_' in name: return name.split('_clipped_')[0]
    if '_clip_' in name:    return name.split('_clip_')[0]
    return name

all_chips  = sorted(CHIPS_DIR.glob('*.tif'))
mask_stems = {m.stem for m in MASKS_DIR.glob('*.tif')}

scene_to_water = defaultdict(list)
scene_to_land  = defaultdict(list)

for chip in all_chips:
    if chip.stem not in mask_stems: continue
    sid = get_scene_id(chip.name)
    if sid in TEST_SCENES: continue
    with rasterio.open(MASKS_DIR / chip.name) as src:
        has_water = (src.read(1) > 0).any()
    if has_water:
        scene_to_water[sid].append(chip)
    else:
        scene_to_land[sid].append(chip)

scene_list = sorted(set(scene_to_water.keys()) | set(scene_to_land.keys()))
random.shuffle(scene_list)

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold1_train_idx, fold1_val_idx = next(iter(kf.split(scene_list)))
train_scenes = set(scene_list[i] for i in fold1_train_idx)
val_scenes   = set(scene_list[i] for i in fold1_val_idx)

print(f'Train scenes: {len(train_scenes)}, Val scenes: {len(val_scenes)}')

train_water = [c for s in train_scenes for c in scene_to_water.get(s, [])]
train_land_all = [c for s in train_scenes for c in scene_to_land.get(s, [])]
n_land = round(len(train_water) * LAND_PCT / (1 - LAND_PCT))
n_land = min(n_land, len(train_land_all))
train_land = random.sample(train_land_all, n_land)
train_chips = train_water + train_land

val_chips = [c for s in val_scenes for c in (scene_to_water.get(s, []) + scene_to_land.get(s, []))]

print(f'Train chips: {len(train_chips):,} (water={len(train_water):,}, land={len(train_land):,})')
print(f'Val chips:   {len(val_chips):,}')

def get_loss_fn(loss_name):
    dice = smp.losses.DiceLoss(mode='binary', from_logits=True, smooth=1.0)
    if loss_name == 'dice':
        return lambda p, t: dice(p, t)
    elif loss_name == 'tversky':
        tversky = smp.losses.TverskyLoss(mode='binary', from_logits=True, alpha=0.3, beta=0.7, smooth=1.0)
        return lambda p, t: tversky(p, t)
    elif loss_name == 'focal_dice':
        focal = smp.losses.FocalLoss(mode='binary')
        return lambda p, t: 0.5 * dice(p, t) + 0.5 * focal(p, t)

def compute_metrics(preds, targets, threshold=0.3):
    pb = (torch.sigmoid(preds) > threshold).cpu().numpy()
    tb = targets.cpu().numpy()
    water_mask = tb.sum(axis=(1,2,3)) > 0
    if water_mask.sum() == 0:
        return {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
    pb = pb[water_mask].flatten().astype(int)
    tb = tb[water_mask].flatten().astype(int)
    return {
        'f1':        round(f1_score(tb, pb, zero_division=0), 4),
        'precision': round(precision_score(tb, pb, zero_division=0), 4),
        'recall':    round(recall_score(tb, pb, zero_division=0), 4),
    }

def compute_multi_threshold_f1(preds, targets, thresholds):
    probs = torch.sigmoid(preds).cpu().numpy()
    tb    = targets.cpu().numpy()
    water_mask = tb.sum(axis=(1,2,3)) > 0
    results = {}
    if water_mask.sum() == 0:
        for t in thresholds:
            results[t] = 0.0
        prob_stats = {'prob_min': 0, 'prob_max': 0, 'prob_mean': 0, 'prob_std': 0}
        return results, prob_stats
    probs_w = probs[water_mask]
    tb_w    = tb[water_mask].flatten().astype(int)
    for t in thresholds:
        pb = (probs_w > t).flatten().astype(int)
        results[t] = round(f1_score(tb_w, pb, zero_division=0), 4)
    prob_stats = {
        'prob_min':  round(float(probs_w.min()), 4),
        'prob_max':  round(float(probs_w.max()), 4),
        'prob_mean': round(float(probs_w.mean()), 4),
        'prob_std':  round(float(probs_w.std()), 4),
    }
    return results, prob_stats

def make_warmup_cosine_scheduler(optimiser, warmup_epochs, max_epochs):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return 0.1 + 0.9 * (epoch / warmup_epochs)
        progress = (epoch - warmup_epochs) / max(1, (max_epochs - warmup_epochs))
        return 0.5 * (1 + np.cos(np.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimiser, lr_lambda)

def train_config(cfg):
    print(f'\n{"="*60}')
    print(f'Config: {cfg["name"]}')
    print(f'  LR={cfg["lr"]}, WD={cfg["wd"]}, Loss={cfg["loss"]}, BS={cfg["bs"]} (warm-up={WARMUP_EPOCHS} epochs)')
    print(f'{"="*60}')

    train_ds = WaterChipDatasetV3(train_chips, MASKS_DIR, mean, std, augment=True)
    val_ds   = WaterChipDatasetV3(val_chips,   MASKS_DIR, mean, std, augment=False)

    train_loader = DataLoader(train_ds, batch_size=cfg['bs'], shuffle=True,
                              num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg['bs'], shuffle=False,
                              num_workers=4, pin_memory=True)

    model = smp.Unet(
        encoder_name='resnet34', encoder_weights='imagenet',
        in_channels=IN_CHANNELS, classes=1, activation=None,
    ).to(DEVICE)

    loss_fn   = get_loss_fn(cfg['loss'])
    optimiser = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    scheduler = make_warmup_cosine_scheduler(optimiser, WARMUP_EPOCHS, MAX_EPOCHS)

    best_val_loss = float('inf')
    no_improve    = 0
    history       = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        t_losses, t_preds, t_targets = [], [], []
        for imgs, masks in tqdm(train_loader, desc=f'{cfg["name"]} E{epoch} train', leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimiser.zero_grad()
            out  = model(imgs)
            loss = loss_fn(out, masks)
            loss.backward()
            optimiser.step()
            t_losses.append(loss.item())
            t_preds.append(out.detach().cpu())
            t_targets.append(masks.detach().cpu())
        scheduler.step()
        current_lr = optimiser.param_groups[0]['lr']
        tm = compute_metrics(torch.cat(t_preds), torch.cat(t_targets))
        del t_preds, t_targets

        model.eval()
        v_losses, v_preds, v_targets = [], [], []
        with torch.no_grad():
            for imgs, masks in tqdm(val_loader, desc=f'{cfg["name"]} E{epoch} val', leave=False):
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                out = model(imgs)
                v_losses.append(loss_fn(out, masks).item())
                v_preds.append(out.cpu())
                v_targets.append(masks.cpu())

        v_preds_cat   = torch.cat(v_preds)
        v_targets_cat = torch.cat(v_targets)
        vm = compute_metrics(v_preds_cat, v_targets_cat)
        multi_f1, prob_stats = compute_multi_threshold_f1(v_preds_cat, v_targets_cat, DIAG_THRESHOLDS)
        del v_preds, v_targets, v_preds_cat, v_targets_cat

        t_loss = round(np.mean(t_losses), 4)
        v_loss = round(np.mean(v_losses), 4)

        history.append({
            'epoch': epoch, 'lr': round(current_lr, 6),
            'train_loss': t_loss, 'val_loss': v_loss,
            'train_f1': tm['f1'], 'val_f1': vm['f1'],
            'val_prec': vm['precision'], 'val_rec': vm['recall'],
            **{f'f1_t{t}': multi_f1[t] for t in DIAG_THRESHOLDS},
            **prob_stats,
        })

        print(f"E{epoch:3d} | LR: {current_lr:.6f} | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} | "
              f"Train F1: {tm['f1']:.4f} | Val F1: {vm['f1']:.4f} | P: {vm['precision']:.4f} | R: {vm['recall']:.4f}")
        print(f"      Probs[min={prob_stats['prob_min']:.4f} max={prob_stats['prob_max']:.4f} "
              f"mean={prob_stats['prob_mean']:.4f} std={prob_stats['prob_std']:.4f}] | "
              f"F1@thresh: " + ' '.join([f't{t}={multi_f1[t]:.3f}' for t in DIAG_THRESHOLDS]))

        if v_loss < best_val_loss:
            best_val_loss = v_loss
            torch.save(model.state_dict(), OUTPUT_DIR / f'{cfg["name"]}_best.pth')
            print(f'  ✓ New best val loss: {best_val_loss:.4f} — model saved')
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  Early stopping at epoch {epoch}')
                break

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(OUTPUT_DIR / f'{cfg["name"]}_history.csv', index=False)

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='Train Loss', color='#2E75B6')
    axes[0].plot(hist_df['epoch'], hist_df['val_loss'],   label='Val Loss',   color='#E8A838')
    axes[0].set_title(f'{cfg["name"]} — Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(hist_df['epoch'], hist_df['train_f1'], label='Train F1', color='#2E75B6')
    axes[1].plot(hist_df['epoch'], hist_df['val_f1'],   label='Val F1',   color='#5DCAA5')
    axes[1].set_title(f'{cfg["name"]} — F1 (t=0.3)'); axes[1].legend(); axes[1].grid(alpha=0.3)
    for t in DIAG_THRESHOLDS:
        axes[2].plot(hist_df['epoch'], hist_df[f'f1_t{t}'], label=f't={t}', alpha=0.8)
    axes[2].set_title(f'{cfg["name"]} — F1 across thresholds'); axes[2].legend(); axes[2].grid(alpha=0.3)
    plt.suptitle(cfg['name'], fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'{cfg["name"]}_curves.png', dpi=150, bbox_inches='tight')
    plt.close()

    return {
        'config': cfg['name'], 'lr': cfg['lr'], 'wd': cfg['wd'],
        'loss': cfg['loss'], 'bs': cfg['bs'],
        'best_val_loss': best_val_loss, 'best_val_f1': hist_df['val_f1'].max(),
    }

# ═══ STABILITY TEST — Lower LR + Tversky ═══
print('\n' + '█'*60); print('STABILITY TEST — Lower peak LR (4e-4) + Tversky + LOW WD (1e-3)'); print('█'*60)
CONFIGS_TEST = [
    {'name': 'Ctest_lr4e4_wd1e3_tversky', 'lr': 4e-4, 'wd': 1e-3, 'loss': 'tversky', 'bs': 32},
]
test_results = [train_config(cfg) for cfg in CONFIGS_TEST]
test_df = pd.DataFrame(test_results)
test_df.to_csv(OUTPUT_DIR / 'stability_test_summary.csv', index=False)
print(test_df[['config','loss','best_val_loss','best_val_f1']].to_string(index=False))

Train scenes: 40, Val scenes: 11
Train chips: 2,606 (water=2,085, land=521)
Val chips:   2,514

████████████████████████████████████████████████████████████
STABILITY TEST — Lower peak LR (4e-4) + Tversky + LOW WD (1e-3)
████████████████████████████████████████████████████████████

Config: Ctest_lr4e4_wd1e3_tversky
  LR=0.0004, WD=0.001, Loss=tversky, BS=32 (warm-up=3 epochs)


E  1 | LR: 0.000160 | Train Loss: 0.7905 | Val Loss: 0.2775 | Train F1: 0.2161 | Val F1: 0.2537 | P: 0.1487 | R: 0.8623
      Probs[min=0.0000 max=0.9952 mean=0.3280 std=0.1712] | F1@thresh: t0.1=0.157 t0.2=0.198 t0.3=0.254 t0.4=0.307 t0.5=0.373
  ✓ New best val loss: 0.2775 — model saved


E  2 | LR: 0.000280 | Train Loss: 0.5666 | Val Loss: 0.2194 | Train F1: 0.4784 | Val F1: 0.7233 | P: 0.7626 | R: 0.6879
      Probs[min=0.0002 max=0.9764 mean=0.1319 std=0.2017] | F1@thresh: t0.1=0.482 t0.2=0.699 t0.3=0.723 t0.4=0.734 t0.5=0.732
  ✓ New best val loss: 0.2194 — model saved


E  3 | LR: 0.000400 | Train Loss: 0.3731 | Val Loss: 0.1886 | Train F1: 0.6762 | Val F1: 0.7096 | P: 0.6778 | R: 0.7445
      Probs[min=0.0002 max=0.9921 mean=0.1023 std=0.2569] | F1@thresh: t0.1=0.687 t0.2=0.705 t0.3=0.710 t0.4=0.712 t0.5=0.714
  ✓ New best val loss: 0.1886 — model saved


E  4 | LR: 0.000400 | Train Loss: 0.3388 | Val Loss: 0.2852 | Train F1: 0.6798 | Val F1: 0.3021 | P: 0.8738 | R: 0.1826
      Probs[min=0.0002 max=0.9997 mean=0.0263 std=0.1233] | F1@thresh: t0.1=0.311 t0.2=0.305 t0.3=0.302 t0.4=0.299 t0.5=0.297


...

E 29 | LR: 0.000333 | Train Loss: 0.1976 | Val Loss: 0.3417 | Train F1: 0.8134 | Val F1: 0.0000 | P: 0.0000 | R: 0.0000
      Probs[min=0.0000 max=0.0800 mean=0.0001 std=0.0001] | F1@thresh: t0.1=0.000 t0.2=0.000 t0.3=0.000 t0.4=0.000 t0.5=0.000


E 30 | LR: 0.000328 | Train Loss: 0.1914 | Val Loss: 0.3415 | Train F1: 0.8242 | Val F1: 0.0013 | P: 0.9654 | R: 0.0007
      Probs[min=0.0000 max=1.0000 mean=0.0001 std=0.0070] | F1@thresh: t0.1=0.002 t0.2=0.001 t0.3=0.001 t0.4=0.001 t0.5=0.001


E 31 | LR: 0.000323 | Train Loss: 0.2050 | Val Loss: 0.3417 | Train F1: 0.8060 | Val F1: 0.0000 | P: 0.0000 | R: 0.0000
      Probs[min=0.0000 max=0.0443 mean=0.0001 std=0.0001] | F1@thresh: t0.1=0.000 t0.2=0.000 t0.3=0.000 t0.4=0.000 t0.5=0.000


E 32 | LR: 0.000318 | Train Loss: 0.1819 | Val Loss: 0.3289 | Train F1: 0.8316 | Val F1: 0.0668 | P: 0.9642 | R: 0.0346
      Probs[min=0.0000 max=1.0000 mean=0.0029 std=0.0529] | F1@thresh: t0.1=0.069 t0.2=0.068 t0.3=0.067 t0.4=0.066 t0.5=0.066


E 33 | LR: 0.000313 | Train Loss: 0.1694 | Val Loss: 0.3194 | Train F1: 0.8376 | Val F1: 0.1214 | P: 0.9909 | R: 0.0647
      Probs[min=0.0000 max=1.0000 mean=0.0053 std=0.0720] | F1@thresh: t0.1=0.124 t0.2=0.122 t0.3=0.121 t0.4=0.121 t0.5=0.120
  Early stopping at epoch 33
                   config    loss  best_val_loss  best_val_f1
Ctest_lr4e4_wd1e3_tversky tversky         0.1787       0.7233


In [ ]:
# ── Cell 3: Stability Test 2 — BCE Loss ──
# NOTE: Do not re-run — results saved to results/unet_training_v3/sweep_bce_test/
#
# Based on evidence from 3 prior runs (original, reduced-aug, diagnostic):
# ALL configurations collapsed into a near-zero-probability degenerate state,
# always at or shortly after LR reached its peak (0.001). This points to
# LR=0.001 being too aggressive for this Dice-loss + WD=0.1 combination,
# regardless of augmentation intensity.
#
# This run tests two changes simultaneously (given time constraints):
# 1. Lower peak LR: 1e-3 -> 4e-4 (60% reduction)
# 2. Tversky loss instead of Dice (different gradient behaviour on imbalanced data,
#    alpha=0.3/beta=0.7 favours recall, may be more stable than Dice here)
#
# Same warm-up (3 epochs), same smoothing (smooth=1.0), same on-the-fly augmentation
# (original intensity, since reduced-aug did NOT prevent collapse either).

import segmentation_models_pytorch as smp
import torch, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import KFold
from collections import defaultdict
from tqdm import tqdm
import rasterio

BASE_DIR   = Path('/mnt/batch/tasks/shared/LS_root/mounts/clusters/v-lmarotti1/code/Users/v-lmarotti/OmniWaterMask')
CHIPS_DIR  = BASE_DIR / 'images/chips_images/images'
MASKS_DIR  = BASE_DIR / 'images/chips_masks/masks'
OUTPUT_DIR = BASE_DIR / 'results/unet_training_v3/sweep_bce_test'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IN_CHANNELS   = 4
MAX_EPOCHS    = 100
PATIENCE      = 15
SEED          = 42
N_FOLDS       = 5
LAND_PCT      = 0.20
WARMUP_EPOCHS = 3
DIAG_THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5]

random.seed(SEED)
np.random.seed(SEED)

with open(BASE_DIR / 'results/unet_training/normalisation_stats.json') as f:
    stats = json.load(f)
mean = np.array(stats['mean'], dtype=np.float32)
std  = np.array(stats['std'],  dtype=np.float32)

TEST_SCENES = {
    '12AUG30193433_sub2', '15JUL02193_sub_2', '16Sep2219_1b_Chiwawa',
    '16Sep2219_1b_Teanaway', 'UGR_2017_08_30_103001007046C600_AOI_1',
    'UGR_2021_07_21_10300100C2972100_AOI_1', 'Wenatchee_2014_08_27_10300100354F7A00_AOI',
    'Wenatchee_2016_08_16_10400100214F5200_AOI_1', 'Wenatchee_2019_07_26_104001004E554F00_AOI_1',
    'Wenatchee_2022_11_09_104001007E867900_AOI_1', 'Wenatchee_2025_02_07_103001010D0A1600_AOI_1',
    'cath_creek_2015_10_09_10300100496A4600_AOI_1', 'cath_creek_2018_12_05_1040010046BF9000_AOI_1',
}

def get_scene_id(filename):
    name = Path(filename).stem
    if '_clipped_' in name: return name.split('_clipped_')[0]
    if '_clip_' in name:    return name.split('_clip_')[0]
    return name

all_chips  = sorted(CHIPS_DIR.glob('*.tif'))
mask_stems = {m.stem for m in MASKS_DIR.glob('*.tif')}

scene_to_water = defaultdict(list)
scene_to_land  = defaultdict(list)

for chip in all_chips:
    if chip.stem not in mask_stems: continue
    sid = get_scene_id(chip.name)
    if sid in TEST_SCENES: continue
    with rasterio.open(MASKS_DIR / chip.name) as src:
        has_water = (src.read(1) > 0).any()
    if has_water:
        scene_to_water[sid].append(chip)
    else:
        scene_to_land[sid].append(chip)

scene_list = sorted(set(scene_to_water.keys()) | set(scene_to_land.keys()))
random.shuffle(scene_list)

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold1_train_idx, fold1_val_idx = next(iter(kf.split(scene_list)))
train_scenes = set(scene_list[i] for i in fold1_train_idx)
val_scenes   = set(scene_list[i] for i in fold1_val_idx)

print(f'Train scenes: {len(train_scenes)}, Val scenes: {len(val_scenes)}')

train_water = [c for s in train_scenes for c in scene_to_water.get(s, [])]
train_land_all = [c for s in train_scenes for c in scene_to_land.get(s, [])]
n_land = round(len(train_water) * LAND_PCT / (1 - LAND_PCT))
n_land = min(n_land, len(train_land_all))
train_land = random.sample(train_land_all, n_land)
train_chips = train_water + train_land

val_chips = [c for s in val_scenes for c in (scene_to_water.get(s, []) + scene_to_land.get(s, []))]

print(f'Train chips: {len(train_chips):,} (water={len(train_water):,}, land={len(train_land):,})')
print(f'Val chips:   {len(val_chips):,}')

def get_loss_fn(loss_name):
    dice = smp.losses.DiceLoss(mode='binary', from_logits=True, smooth=1.0)
    if loss_name == 'dice':
        return lambda p, t: dice(p, t)
    elif loss_name == 'tversky':
        tversky = smp.losses.TverskyLoss(mode='binary', from_logits=True, alpha=0.3, beta=0.7, smooth=1.0)
        return lambda p, t: tversky(p, t)
    elif loss_name == 'focal_dice':
        focal = smp.losses.FocalLoss(mode='binary')
        return lambda p, t: 0.5 * dice(p, t) + 0.5 * focal(p, t)
    elif loss_name == 'bce':
        bce = torch.nn.BCEWithLogitsLoss()
        return lambda p, t: bce(p, t)

def compute_metrics(preds, targets, threshold=0.3):
    pb = (torch.sigmoid(preds) > threshold).cpu().numpy()
    tb = targets.cpu().numpy()
    water_mask = tb.sum(axis=(1,2,3)) > 0
    if water_mask.sum() == 0:
        return {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
    pb = pb[water_mask].flatten().astype(int)
    tb = tb[water_mask].flatten().astype(int)
    return {
        'f1':        round(f1_score(tb, pb, zero_division=0), 4),
        'precision': round(precision_score(tb, pb, zero_division=0), 4),
        'recall':    round(recall_score(tb, pb, zero_division=0), 4),
    }

def compute_multi_threshold_f1(preds, targets, thresholds):
    probs = torch.sigmoid(preds).cpu().numpy()
    tb    = targets.cpu().numpy()
    water_mask = tb.sum(axis=(1,2,3)) > 0
    results = {}
    if water_mask.sum() == 0:
        for t in thresholds:
            results[t] = 0.0
        prob_stats = {'prob_min': 0, 'prob_max': 0, 'prob_mean': 0, 'prob_std': 0}
        return results, prob_stats
    probs_w = probs[water_mask]
    tb_w    = tb[water_mask].flatten().astype(int)
    for t in thresholds:
        pb = (probs_w > t).flatten().astype(int)
        results[t] = round(f1_score(tb_w, pb, zero_division=0), 4)
    prob_stats = {
        'prob_min':  round(float(probs_w.min()), 4),
        'prob_max':  round(float(probs_w.max()), 4),
        'prob_mean': round(float(probs_w.mean()), 4),
        'prob_std':  round(float(probs_w.std()), 4),
    }
    return results, prob_stats

def make_warmup_cosine_scheduler(optimiser, warmup_epochs, max_epochs):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return 0.1 + 0.9 * (epoch / warmup_epochs)
        progress = (epoch - warmup_epochs) / max(1, (max_epochs - warmup_epochs))
        return 0.5 * (1 + np.cos(np.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimiser, lr_lambda)

def train_config(cfg):
    print(f'\n{"="*60}')
    print(f'Config: {cfg["name"]}')
    print(f'  LR={cfg["lr"]}, WD={cfg["wd"]}, Loss={cfg["loss"]}, BS={cfg["bs"]} (warm-up={WARMUP_EPOCHS} epochs)')
    print(f'{"="*60}')

    train_ds = WaterChipDatasetV3(train_chips, MASKS_DIR, mean, std, augment=True)
    val_ds   = WaterChipDatasetV3(val_chips,   MASKS_DIR, mean, std, augment=False)

    train_loader = DataLoader(train_ds, batch_size=cfg['bs'], shuffle=True,
                              num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg['bs'], shuffle=False,
                              num_workers=4, pin_memory=True)

    model = smp.Unet(
        encoder_name='resnet34', encoder_weights='imagenet',
        in_channels=IN_CHANNELS, classes=1, activation=None,
    ).to(DEVICE)

    loss_fn   = get_loss_fn(cfg['loss'])
    optimiser = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    scheduler = make_warmup_cosine_scheduler(optimiser, WARMUP_EPOCHS, MAX_EPOCHS)

    best_val_loss = float('inf')
    no_improve    = 0
    history       = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        t_losses, t_preds, t_targets = [], [], []
        for imgs, masks in tqdm(train_loader, desc=f'{cfg["name"]} E{epoch} train', leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimiser.zero_grad()
            out  = model(imgs)
            loss = loss_fn(out, masks)
            loss.backward()
            optimiser.step()
            t_losses.append(loss.item())
            t_preds.append(out.detach().cpu())
            t_targets.append(masks.detach().cpu())
        scheduler.step()
        current_lr = optimiser.param_groups[0]['lr']
        tm = compute_metrics(torch.cat(t_preds), torch.cat(t_targets))
        del t_preds, t_targets

        model.eval()
        v_losses, v_preds, v_targets = [], [], []
        with torch.no_grad():
            for imgs, masks in tqdm(val_loader, desc=f'{cfg["name"]} E{epoch} val', leave=False):
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                out = model(imgs)
                v_losses.append(loss_fn(out, masks).item())
                v_preds.append(out.cpu())
                v_targets.append(masks.cpu())

        v_preds_cat   = torch.cat(v_preds)
        v_targets_cat = torch.cat(v_targets)
        vm = compute_metrics(v_preds_cat, v_targets_cat)
        multi_f1, prob_stats = compute_multi_threshold_f1(v_preds_cat, v_targets_cat, DIAG_THRESHOLDS)
        del v_preds, v_targets, v_preds_cat, v_targets_cat

        t_loss = round(np.mean(t_losses), 4)
        v_loss = round(np.mean(v_losses), 4)

        history.append({
            'epoch': epoch, 'lr': round(current_lr, 6),
            'train_loss': t_loss, 'val_loss': v_loss,
            'train_f1': tm['f1'], 'val_f1': vm['f1'],
            'val_prec': vm['precision'], 'val_rec': vm['recall'],
            **{f'f1_t{t}': multi_f1[t] for t in DIAG_THRESHOLDS},
            **prob_stats,
        })

        print(f"E{epoch:3d} | LR: {current_lr:.6f} | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} | "
              f"Train F1: {tm['f1']:.4f} | Val F1: {vm['f1']:.4f} | P: {vm['precision']:.4f} | R: {vm['recall']:.4f}")
        print(f"      Probs[min={prob_stats['prob_min']:.4f} max={prob_stats['prob_max']:.4f} "
              f"mean={prob_stats['prob_mean']:.4f} std={prob_stats['prob_std']:.4f}] | "
              f"F1@thresh: " + ' '.join([f't{t}={multi_f1[t]:.3f}' for t in DIAG_THRESHOLDS]))

        if v_loss < best_val_loss:
            best_val_loss = v_loss
            torch.save(model.state_dict(), OUTPUT_DIR / f'{cfg["name"]}_best.pth')
            print(f'  ✓ New best val loss: {best_val_loss:.4f} — model saved')
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  Early stopping at epoch {epoch}')
                break

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(OUTPUT_DIR / f'{cfg["name"]}_history.csv', index=False)

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='Train Loss', color='#2E75B6')
    axes[0].plot(hist_df['epoch'], hist_df['val_loss'],   label='Val Loss',   color='#E8A838')
    axes[0].set_title(f'{cfg["name"]} — Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(hist_df['epoch'], hist_df['train_f1'], label='Train F1', color='#2E75B6')
    axes[1].plot(hist_df['epoch'], hist_df['val_f1'],   label='Val F1',   color='#5DCAA5')
    axes[1].set_title(f'{cfg["name"]} — F1 (t=0.3)'); axes[1].legend(); axes[1].grid(alpha=0.3)
    for t in DIAG_THRESHOLDS:
        axes[2].plot(hist_df['epoch'], hist_df[f'f1_t{t}'], label=f't={t}', alpha=0.8)
    axes[2].set_title(f'{cfg["name"]} — F1 across thresholds'); axes[2].legend(); axes[2].grid(alpha=0.3)
    plt.suptitle(cfg['name'], fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'{cfg["name"]}_curves.png', dpi=150, bbox_inches='tight')
    plt.close()

    return {
        'config': cfg['name'], 'lr': cfg['lr'], 'wd': cfg['wd'],
        'loss': cfg['loss'], 'bs': cfg['bs'],
        'best_val_loss': best_val_loss, 'best_val_f1': hist_df['val_f1'].max(),
    }

# ═══ STABILITY TEST — Lower LR + Tversky ═══
print('\n' + '█'*60); print('STABILITY TEST — BCE loss (instead of Tversky/Dice)'); print('█'*60)
CONFIGS_TEST = [
    {'name': 'Ctest_lr4e4_wd1e3_bce', 'lr': 4e-4, 'wd': 1e-3, 'loss': 'bce', 'bs': 32},
]
test_results = [train_config(cfg) for cfg in CONFIGS_TEST]
test_df = pd.DataFrame(test_results)
test_df.to_csv(OUTPUT_DIR / 'stability_test_summary.csv', index=False)
print(test_df[['config','loss','best_val_loss','best_val_f1']].to_string(index=False))

Train scenes: 40, Val scenes: 11
Train chips: 2,606 (water=2,085, land=521)
Val chips:   2,514

████████████████████████████████████████████████████████████
STABILITY TEST — BCE loss (instead of Tversky/Dice)
████████████████████████████████████████████████████████████

Config: Ctest_lr4e4_wd1e3_bce
  LR=0.0004, WD=0.001, Loss=bce, BS=32 (warm-up=3 epochs)


E  1 | LR: 0.000160 | Train Loss: 0.6375 | Val Loss: 0.5421 | Train F1: 0.1708 | Val F1: 0.1543 | P: 0.0842 | R: 0.9274
      Probs[min=0.0000 max=1.0000 mean=0.3467 std=0.0801] | F1@thresh: t0.1=0.152 t0.2=0.152 t0.3=0.154 t0.4=0.103 t0.5=0.032
  ✓ New best val loss: 0.5421 — model saved


E  2 | LR: 0.000280 | Train Loss: 0.3515 | Val Loss: 0.2251 | Train F1: 0.3703 | Val F1: 0.5870 | P: 0.5947 | R: 0.5795
      Probs[min=0.0045 max=1.0000 mean=0.2095 std=0.0972] | F1@thresh: t0.1=0.152 t0.2=0.279 t0.3=0.587 t0.4=0.544 t0.5=0.395
  ✓ New best val loss: 0.2251 — model saved


E  3 | LR: 0.000400 | Train Loss: 0.2276 | Val Loss: 0.1430 | Train F1: 0.6485 | Val F1: 0.5637 | P: 0.8887 | R: 0.4128
      Probs[min=0.0010 max=1.0000 mean=0.1361 std=0.1215] | F1@thresh: t0.1=0.254 t0.2=0.592 t0.3=0.564 t0.4=0.535 t0.5=0.503
  ✓ New best val loss: 0.1430 — model saved


E  4 | LR: 0.000400 | Train Loss: 0.1579 | Val Loss: 0.1032 | Train F1: 0.7249 | Val F1: 0.0947 | P: 0.9786 | R: 0.0498
      Probs[min=0.0019 max=1.0000 mean=0.0560 std=0.0362] | F1@thresh: t0.1=0.217 t0.2=0.121 t0.3=0.095 t0.4=0.075 t0.5=0.041
  ✓ New best val loss: 0.1032 — model saved


...

E  8 | LR: 0.000397 | Train Loss: 0.0908 | Val Loss: 0.0668 | Train F1: 0.7727 | Val F1: 0.1690 | P: 0.9237 | R: 0.0930
      Probs[min=0.0009 max=0.7831 mean=0.0226 std=0.0493] | F1@thresh: t0.1=0.526 t0.2=0.361 t0.3=0.169 t0.4=0.096 t0.5=0.025
  ✓ New best val loss: 0.0668 — model saved


E  9 | LR: 0.000396 | Train Loss: 0.0761 | Val Loss: 0.0750 | Train F1: 0.8187 | Val F1: 0.0840 | P: 0.9983 | R: 0.0438
      Probs[min=0.0009 max=0.9017 mean=0.0154 std=0.0340] | F1@thresh: t0.1=0.344 t0.2=0.184 t0.3=0.084 t0.4=0.039 t0.5=0.011


E 10 | LR: 0.000395 | Train Loss: 0.0735 | Val Loss: 0.0891 | Train F1: 0.8145 | Val F1: 0.0011 | P: 0.9979 | R: 0.0006
      Probs[min=0.0002 max=0.5321 mean=0.0079 std=0.0092] | F1@thresh: t0.1=0.024 t0.2=0.006 t0.3=0.001 t0.4=0.000 t0.5=0.000


E 11 | LR: 0.000393 | Train Loss: 0.0774 | Val Loss: 0.0920 | Train F1: 0.8104 | Val F1: 0.0007 | P: 0.9193 | R: 0.0003
      Probs[min=0.0010 max=0.5334 mean=0.0070 std=0.0101] | F1@thresh: t0.1=0.042 t0.2=0.004 t0.3=0.001 t0.4=0.000 t0.5=0.000


Ctest_lr4e4_wd1e3_bce E12 train:  68%|██████▊   | 56/82 [03:06<00:44,  1.70s/it]

In [3]:
# ── Cell 4: Stability Test 3 — LR=1e-4 + Focal+Dice (winner → used in final sweep) ──
# NOTE: Do not re-run — results saved to results/unet_training_v3/sweep_lr1e4_test/
#
# Based on evidence from 3 prior runs (original, reduced-aug, diagnostic):
# ALL configurations collapsed into a near-zero-probability degenerate state,
# always at or shortly after LR reached its peak (0.001). This points to
# LR=0.001 being too aggressive for this Dice-loss + WD=0.1 combination,
# regardless of augmentation intensity.
#
# This run tests two changes simultaneously (given time constraints):
# 1. Lower peak LR: 1e-3 -> 4e-4 (60% reduction)
# 2. Tversky loss instead of Dice (different gradient behaviour on imbalanced data,
#    alpha=0.3/beta=0.7 favours recall, may be more stable than Dice here)
#
# Same warm-up (3 epochs), same smoothing (smooth=1.0), same on-the-fly augmentation
# (original intensity, since reduced-aug did NOT prevent collapse either).

import segmentation_models_pytorch as smp
import torch, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import KFold
from collections import defaultdict
from tqdm import tqdm
import rasterio

BASE_DIR   = Path('/mnt/batch/tasks/shared/LS_root/mounts/clusters/v-lmarotti1/code/Users/v-lmarotti/OmniWaterMask')
CHIPS_DIR  = BASE_DIR / 'images/chips_images/images'
MASKS_DIR  = BASE_DIR / 'images/chips_masks/masks'
OUTPUT_DIR = BASE_DIR / 'results/unet_training_v3/sweep_lr1e4_test'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IN_CHANNELS   = 4
MAX_EPOCHS    = 100
PATIENCE      = 15
SEED          = 42
N_FOLDS       = 5
LAND_PCT      = 0.20
WARMUP_EPOCHS = 3
DIAG_THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5]

random.seed(SEED)
np.random.seed(SEED)

with open(BASE_DIR / 'results/unet_training/normalisation_stats.json') as f:
    stats = json.load(f)
mean = np.array(stats['mean'], dtype=np.float32)
std  = np.array(stats['std'],  dtype=np.float32)

TEST_SCENES = {
    '12AUG30193433_sub2', '15JUL02193_sub_2', '16Sep2219_1b_Chiwawa',
    '16Sep2219_1b_Teanaway', 'UGR_2017_08_30_103001007046C600_AOI_1',
    'UGR_2021_07_21_10300100C2972100_AOI_1', 'Wenatchee_2014_08_27_10300100354F7A00_AOI',
    'Wenatchee_2016_08_16_10400100214F5200_AOI_1', 'Wenatchee_2019_07_26_104001004E554F00_AOI_1',
    'Wenatchee_2022_11_09_104001007E867900_AOI_1', 'Wenatchee_2025_02_07_103001010D0A1600_AOI_1',
    'cath_creek_2015_10_09_10300100496A4600_AOI_1', 'cath_creek_2018_12_05_1040010046BF9000_AOI_1',
}

def get_scene_id(filename):
    name = Path(filename).stem
    if '_clipped_' in name: return name.split('_clipped_')[0]
    if '_clip_' in name:    return name.split('_clip_')[0]
    return name

all_chips  = sorted(CHIPS_DIR.glob('*.tif'))
mask_stems = {m.stem for m in MASKS_DIR.glob('*.tif')}

scene_to_water = defaultdict(list)
scene_to_land  = defaultdict(list)

for chip in all_chips:
    if chip.stem not in mask_stems: continue
    sid = get_scene_id(chip.name)
    if sid in TEST_SCENES: continue
    with rasterio.open(MASKS_DIR / chip.name) as src:
        has_water = (src.read(1) > 0).any()
    if has_water:
        scene_to_water[sid].append(chip)
    else:
        scene_to_land[sid].append(chip)

scene_list = sorted(set(scene_to_water.keys()) | set(scene_to_land.keys()))
random.shuffle(scene_list)

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold1_train_idx, fold1_val_idx = next(iter(kf.split(scene_list)))
train_scenes = set(scene_list[i] for i in fold1_train_idx)
val_scenes   = set(scene_list[i] for i in fold1_val_idx)

print(f'Train scenes: {len(train_scenes)}, Val scenes: {len(val_scenes)}')

train_water = [c for s in train_scenes for c in scene_to_water.get(s, [])]
train_land_all = [c for s in train_scenes for c in scene_to_land.get(s, [])]
n_land = round(len(train_water) * LAND_PCT / (1 - LAND_PCT))
n_land = min(n_land, len(train_land_all))
train_land = random.sample(train_land_all, n_land)
train_chips = train_water + train_land

val_chips = [c for s in val_scenes for c in (scene_to_water.get(s, []) + scene_to_land.get(s, []))]

print(f'Train chips: {len(train_chips):,} (water={len(train_water):,}, land={len(train_land):,})')
print(f'Val chips:   {len(val_chips):,}')

def get_loss_fn(loss_name):
    dice = smp.losses.DiceLoss(mode='binary', from_logits=True, smooth=1.0)
    if loss_name == 'dice':
        return lambda p, t: dice(p, t)
    elif loss_name == 'tversky':
        tversky = smp.losses.TverskyLoss(mode='binary', from_logits=True, alpha=0.3, beta=0.7, smooth=1.0)
        return lambda p, t: tversky(p, t)
    elif loss_name == 'focal_dice':
        focal = smp.losses.FocalLoss(mode='binary')
        return lambda p, t: 0.5 * dice(p, t) + 0.5 * focal(p, t)

def compute_metrics(preds, targets, threshold=0.3):
    pb = (torch.sigmoid(preds) > threshold).cpu().numpy()
    tb = targets.cpu().numpy()
    water_mask = tb.sum(axis=(1,2,3)) > 0
    if water_mask.sum() == 0:
        return {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
    pb = pb[water_mask].flatten().astype(int)
    tb = tb[water_mask].flatten().astype(int)
    return {
        'f1':        round(f1_score(tb, pb, zero_division=0), 4),
        'precision': round(precision_score(tb, pb, zero_division=0), 4),
        'recall':    round(recall_score(tb, pb, zero_division=0), 4),
    }

def compute_multi_threshold_f1(preds, targets, thresholds):
    probs = torch.sigmoid(preds).cpu().numpy()
    tb    = targets.cpu().numpy()
    water_mask = tb.sum(axis=(1,2,3)) > 0
    results = {}
    if water_mask.sum() == 0:
        for t in thresholds:
            results[t] = 0.0
        prob_stats = {'prob_min': 0, 'prob_max': 0, 'prob_mean': 0, 'prob_std': 0}
        return results, prob_stats
    probs_w = probs[water_mask]
    tb_w    = tb[water_mask].flatten().astype(int)
    for t in thresholds:
        pb = (probs_w > t).flatten().astype(int)
        results[t] = round(f1_score(tb_w, pb, zero_division=0), 4)
    prob_stats = {
        'prob_min':  round(float(probs_w.min()), 4),
        'prob_max':  round(float(probs_w.max()), 4),
        'prob_mean': round(float(probs_w.mean()), 4),
        'prob_std':  round(float(probs_w.std()), 4),
    }
    return results, prob_stats

def make_warmup_cosine_scheduler(optimiser, warmup_epochs, max_epochs):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return 0.1 + 0.9 * (epoch / warmup_epochs)
        progress = (epoch - warmup_epochs) / max(1, (max_epochs - warmup_epochs))
        return 0.5 * (1 + np.cos(np.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimiser, lr_lambda)

def train_config(cfg):
    print(f'\n{"="*60}')
    print(f'Config: {cfg["name"]}')
    print(f'  LR={cfg["lr"]}, WD={cfg["wd"]}, Loss={cfg["loss"]}, BS={cfg["bs"]} (warm-up={WARMUP_EPOCHS} epochs)')
    print(f'{"="*60}')

    train_ds = WaterChipDatasetV3(train_chips, MASKS_DIR, mean, std, augment=True)
    val_ds   = WaterChipDatasetV3(val_chips,   MASKS_DIR, mean, std, augment=False)

    train_loader = DataLoader(train_ds, batch_size=cfg['bs'], shuffle=True,
                              num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg['bs'], shuffle=False,
                              num_workers=4, pin_memory=True)

    model = smp.Unet(
        encoder_name='resnet34', encoder_weights='imagenet',
        in_channels=IN_CHANNELS, classes=1, activation=None,
    ).to(DEVICE)

    loss_fn   = get_loss_fn(cfg['loss'])
    optimiser = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])
    scheduler = make_warmup_cosine_scheduler(optimiser, WARMUP_EPOCHS, MAX_EPOCHS)

    best_val_loss = float('inf')
    no_improve    = 0
    history       = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        t_losses, t_preds, t_targets = [], [], []
        for imgs, masks in tqdm(train_loader, desc=f'{cfg["name"]} E{epoch} train', leave=False):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimiser.zero_grad()
            out  = model(imgs)
            loss = loss_fn(out, masks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimiser.step()
            t_losses.append(loss.item())
            t_preds.append(out.detach().cpu())
            t_targets.append(masks.detach().cpu())
        scheduler.step()
        current_lr = optimiser.param_groups[0]['lr']
        tm = compute_metrics(torch.cat(t_preds), torch.cat(t_targets))
        del t_preds, t_targets

        model.eval()
        v_losses, v_preds, v_targets = [], [], []
        with torch.no_grad():
            for imgs, masks in tqdm(val_loader, desc=f'{cfg["name"]} E{epoch} val', leave=False):
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                out = model(imgs)
                v_losses.append(loss_fn(out, masks).item())
                v_preds.append(out.cpu())
                v_targets.append(masks.cpu())

        v_preds_cat   = torch.cat(v_preds)
        v_targets_cat = torch.cat(v_targets)
        vm = compute_metrics(v_preds_cat, v_targets_cat)
        multi_f1, prob_stats = compute_multi_threshold_f1(v_preds_cat, v_targets_cat, DIAG_THRESHOLDS)
        del v_preds, v_targets, v_preds_cat, v_targets_cat

        t_loss = round(np.mean(t_losses), 4)
        v_loss = round(np.mean(v_losses), 4)

        history.append({
            'epoch': epoch, 'lr': round(current_lr, 6),
            'train_loss': t_loss, 'val_loss': v_loss,
            'train_f1': tm['f1'], 'val_f1': vm['f1'],
            'val_prec': vm['precision'], 'val_rec': vm['recall'],
            **{f'f1_t{t}': multi_f1[t] for t in DIAG_THRESHOLDS},
            **prob_stats,
        })

        print(f"E{epoch:3d} | LR: {current_lr:.6f} | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} | "
              f"Train F1: {tm['f1']:.4f} | Val F1: {vm['f1']:.4f} | P: {vm['precision']:.4f} | R: {vm['recall']:.4f}")
        print(f"      Probs[min={prob_stats['prob_min']:.4f} max={prob_stats['prob_max']:.4f} "
              f"mean={prob_stats['prob_mean']:.4f} std={prob_stats['prob_std']:.4f}] | "
              f"F1@thresh: " + ' '.join([f't{t}={multi_f1[t]:.3f}' for t in DIAG_THRESHOLDS]))

        if v_loss < best_val_loss:
            best_val_loss = v_loss
            torch.save(model.state_dict(), OUTPUT_DIR / f'{cfg["name"]}_best.pth')
            print(f'  ✓ New best val loss: {best_val_loss:.4f} — model saved')
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  Early stopping at epoch {epoch}')
                break

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(OUTPUT_DIR / f'{cfg["name"]}_history.csv', index=False)

    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='Train Loss', color='#2E75B6')
    axes[0].plot(hist_df['epoch'], hist_df['val_loss'],   label='Val Loss',   color='#E8A838')
    axes[0].set_title(f'{cfg["name"]} — Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(hist_df['epoch'], hist_df['train_f1'], label='Train F1', color='#2E75B6')
    axes[1].plot(hist_df['epoch'], hist_df['val_f1'],   label='Val F1',   color='#5DCAA5')
    axes[1].set_title(f'{cfg["name"]} — F1 (t=0.3)'); axes[1].legend(); axes[1].grid(alpha=0.3)
    for t in DIAG_THRESHOLDS:
        axes[2].plot(hist_df['epoch'], hist_df[f'f1_t{t}'], label=f't={t}', alpha=0.8)
    axes[2].set_title(f'{cfg["name"]} — F1 across thresholds'); axes[2].legend(); axes[2].grid(alpha=0.3)
    plt.suptitle(cfg['name'], fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'{cfg["name"]}_curves.png', dpi=150, bbox_inches='tight')
    plt.close()

    return {
        'config': cfg['name'], 'lr': cfg['lr'], 'wd': cfg['wd'],
        'loss': cfg['loss'], 'bs': cfg['bs'],
        'best_val_loss': best_val_loss, 'best_val_f1': hist_df['val_f1'].max(),
    }

# ═══ STABILITY TEST — Lower LR + Tversky ═══
print('\n' + '█'*60); print('LR TEST — LR=1e-4 + Focal+Dice (vs C4 LR=2e-4)'); print('█'*60)
CONFIGS_TEST = [
    {'name': 'Ctest_lr1e4_focal_dice', 'lr': 1e-4, 'wd': 1e-3, 'loss': 'focal_dice', 'bs': 64},
]
test_results = [train_config(cfg) for cfg in CONFIGS_TEST]
test_df = pd.DataFrame(test_results)
test_df.to_csv(OUTPUT_DIR / 'stability_test_summary.csv', index=False)
print(test_df[['config','loss','best_val_loss','best_val_f1']].to_string(index=False))

Train scenes: 40, Val scenes: 11
Train chips: 2,606 (water=2,085, land=521)
Val chips:   2,514

████████████████████████████████████████████████████████████
LR TEST — LR=1e-4 + Focal+Dice (vs C4 LR=2e-4)
████████████████████████████████████████████████████████████

Config: Ctest_lr1e4_focal_dice
  LR=0.0001, WD=0.001, Loss=focal_dice, BS=64 (warm-up=3 epochs)


Ctest_lr1e4_focal_dice E1 train:   2%|▏         | 1/41 [00:20<13:56, 20.90s/it]/anaconda/envs/owm/lib/python3.12/site-packages/torch/autograd/graph.py:744: UserWarning: Plan failed with an OutOfMemoryError: CUDA out of memory. Tried to allocate 2.39 GiB. GPU  (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:924.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
Ctest_lr1e4_focal_dice E1 train:  98%|█████████▊| 40/41 [03:19<00:02,  2.91s/it]/anaconda/envs/owm/lib/python3.12/site-packages/torch/autograd/graph.py:744: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
                                                                   

E  1 | LR: 0.000040 | Train Loss: 0.5096 | Val Loss: 0.2578 | Train F1: 0.1754 | Val F1: 0.1598 | P: 0.0872 | R: 0.9519
      Probs[min=0.0000 max=1.0000 mean=0.4271 std=0.1043] | F1@thresh: t0.1=0.153 t0.2=0.155 t0.3=0.160 t0.4=0.170 t0.5=0.149
  ✓ New best val loss: 0.2578 — model saved


E  2 | LR: 0.000070 | Train Loss: 0.4608 | Val Loss: 0.2107 | Train F1: 0.2276 | Val F1: 0.2964 | P: 0.1804 | R: 0.8302
      Probs[min=0.0000 max=1.0000 mean=0.3301 std=0.1669] | F1@thresh: t0.1=0.154 t0.2=0.162 t0.3=0.296 t0.4=0.420 t0.5=0.535
  ✓ New best val loss: 0.2107 — model saved


E  3 | LR: 0.000100 | Train Loss: 0.4006 | Val Loss: 0.1864 | Train F1: 0.4254 | Val F1: 0.6035 | P: 0.5097 | R: 0.7396
      Probs[min=0.0000 max=1.0000 mean=0.2353 std=0.1688] | F1@thresh: t0.1=0.154 t0.2=0.342 t0.3=0.604 t0.4=0.673 t0.5=0.695
  ✓ New best val loss: 0.1864 — model saved


...

E 61 | LR: 0.000035 | Train Loss: 0.0619 | Val Loss: 0.0574 | Train F1: 0.9203 | Val F1: 0.8650 | P: 0.9383 | R: 0.8023
      Probs[min=0.0000 max=1.0000 mean=0.0681 std=0.2415] | F1@thresh: t0.1=0.869 t0.2=0.867 t0.3=0.865 t0.4=0.861 t0.5=0.857


E 62 | LR: 0.000033 | Train Loss: 0.0638 | Val Loss: 0.0496 | Train F1: 0.9206 | Val F1: 0.8780 | P: 0.9167 | R: 0.8424
      Probs[min=0.0000 max=1.0000 mean=0.0734 std=0.2512] | F1@thresh: t0.1=0.878 t0.2=0.878 t0.3=0.878 t0.4=0.876 t0.5=0.875


E 63 | LR: 0.000032 | Train Loss: 0.0607 | Val Loss: 0.0507 | Train F1: 0.9272 | Val F1: 0.8848 | P: 0.9149 | R: 0.8565
      Probs[min=0.0000 max=1.0000 mean=0.0745 std=0.2524] | F1@thresh: t0.1=0.884 t0.2=0.885 t0.3=0.885 t0.4=0.883 t0.5=0.880


E 64 | LR: 0.000030 | Train Loss: 0.0628 | Val Loss: 0.0935 | Train F1: 0.9229 | Val F1: 0.7902 | P: 0.9236 | R: 0.6904
      Probs[min=0.0000 max=0.9999 mean=0.0575 std=0.2171] | F1@thresh: t0.1=0.811 t0.2=0.800 t0.3=0.790 t0.4=0.780 t0.5=0.770


E 65 | LR: 0.000029 | Train Loss: 0.0601 | Val Loss: 0.0518 | Train F1: 0.9280 | Val F1: 0.8694 | P: 0.9314 | R: 0.8152
      Probs[min=0.0000 max=1.0000 mean=0.0701 std=0.2458] | F1@thresh: t0.1=0.873 t0.2=0.871 t0.3=0.869 t0.4=0.867 t0.5=0.864
  Early stopping at epoch 65
                config       loss  best_val_loss  best_val_f1
Ctest_lr1e4_focal_dice focal_dice         0.0488       0.8916
